# 03 — Build the clean spine (Phases 3–4)

**Phase 3:** aggregate Pudding characters → film grain, attach `imdb_id` via
`meta_data7` on `script_id`. Output: `data/interim/pudding_film.csv`.

**Phase 4:** normalize Bechdel `imdb_id`, inner-join to the aggregated Pudding
table on `imdb_id`, filter to 1990–2015. Output: `data/interim/spine.csv`.

This is the CLEAN join (shared key, no fuzzy matching) — the plan says it
should be near-lossless *among films Bechdel covers*, and we validate that
here before touching the fuzzy Numbers join (Phase 5).

**Inputs (read-only):** `pudding_character_list5.csv`, `pudding_meta_data7.csv`,
`bechdel_tidytuesday_2021.csv` (mirror — see `data/raw/README_provenance.md`).

In [1]:
from pathlib import Path

import pandas as pd

RAW = Path.cwd().parent / "data" / "raw"
INTERIM = Path.cwd().parent / "data" / "interim"

chars = pd.read_csv(RAW / "pudding_character_list5.csv", encoding="latin-1")
meta = pd.read_csv(RAW / "pudding_meta_data7.csv", encoding="latin-1")
bechdel = pd.read_csv(RAW / "bechdel_tidytuesday_2021.csv", dtype={"imdb_id": "string"})

print(f"chars {chars.shape} | meta {meta.shape} | bechdel {bechdel.shape}")

chars (23048, 5) | meta (2000, 6) | bechdel (8839, 5)


## Phase 3a — clean the character table

Documented exclusions, applied once, counted:
1. `gender == '?'` (5 rows) — unknowable, dropped
2. `words <= 0` or missing — cannot contribute to dialogue shares

In [2]:
n0 = len(chars)
chars = chars[chars["gender"].isin(["m", "f"])]
n1 = len(chars)
chars = chars[chars["words"] > 0]
n2 = len(chars)
print(f"dropped {n0 - n1} gender-'?' rows, {n1 - n2} zero/negative-word rows -> {n2:,} characters")

dropped 5 gender-'?' rows, 0 zero/negative-word rows -> 23,043 characters


## Phase 3b — aggregate to film grain

Per `script_id`:
- `pct_female_dialogue` — female words / total words (the headline predictor)
- `pct_female_characters` — share of speaking characters coded female
- `female_lead` — is the top-words character female?
- `n_characters`, `total_words` — size/denominator context

In [3]:
chars = chars.assign(is_f=chars["gender"].eq("f"),
                     f_words=lambda d: d["words"].where(d["is_f"], 0))

agg = chars.groupby("script_id").agg(
    total_words=("words", "sum"),
    f_words=("f_words", "sum"),
    n_characters=("words", "size"),
    n_female=("is_f", "sum"),
)
agg["pct_female_dialogue"] = agg["f_words"] / agg["total_words"]
agg["pct_female_characters"] = agg["n_female"] / agg["n_characters"]

# female_lead: gender of the character with the most words in each film
lead_idx = chars.groupby("script_id")["words"].idxmax()
agg["female_lead"] = chars.loc[lead_idx].set_index("script_id")["is_f"]

agg = agg.reset_index()
print(f"{len(agg):,} films aggregated")
agg[["pct_female_dialogue", "pct_female_characters", "n_characters"]].describe().round(3)

2,000 films aggregated


,pct_female_dialogue,pct_female_characters,n_characters
count,2000.000,2000.000,2000.000
mean,0.283,0.310,11.522
std,0.197,0.164,4.461
min,0.000,0.000,2.000
25%,0.135,0.190,8.000
50%,0.243,0.300,11.000
75%,0.389,0.412,14.000
max,1.000,1.000,40.000


## Phase 3c — attach film identity from meta_data7

`script_id` → `imdb_id`, `title`, `year`, Pudding's inflation-adjusted `gross`.
Key hygiene first: `imdb_id` must be unique in meta before it can serve as the
Phase 4 join key — duplicates are inspected and resolved explicitly.

In [4]:
print("script_id unique in meta:", meta["script_id"].is_unique)
print("imdb_id unique in meta:", meta["imdb_id"].is_unique)

dupes = meta[meta["imdb_id"].duplicated(keep=False)].sort_values("imdb_id")
print(f"rows sharing an imdb_id: {len(dupes)}")
if len(dupes):
    display(dupes[["script_id", "imdb_id", "title", "year", "gross"]])
    # same film scraped twice -> keep the row whose script has more total words
    # (fuller transcript), merging aggregates by script_id first
    meta_agg = meta.merge(agg[["script_id", "total_words"]], on="script_id", how="left")
    keep_ids = meta_agg.sort_values("total_words", ascending=False)\
        .drop_duplicates("imdb_id", keep="first")["script_id"]
    meta = meta[meta["script_id"].isin(keep_ids)]
    print(f"resolved: kept fullest transcript per imdb_id -> {len(meta):,} meta rows")

script_id unique in meta: True
imdb_id unique in meta: True
rows sharing an imdb_id: 0


In [5]:
pudding_film = meta[["script_id", "imdb_id", "title", "year", "gross"]]\
    .rename(columns={"gross": "gross_infl_adj_musd"})\
    .merge(agg, on="script_id", how="inner")

assert pudding_film["imdb_id"].is_unique
print(f"pudding_film: {pudding_film.shape[0]:,} rows x {pudding_film.shape[1]} cols")

pudding_film.to_csv(INTERIM / "pudding_film.csv", index=False)
print("wrote ->", INTERIM / "pudding_film.csv")
pudding_film.head(3)

pudding_film: 2,000 rows x 12 cols
wrote -> /Users/trevorjmckee/Documents/Work/School/03_Summer_2026/DSC530/final_project/data/interim/pudding_film.csv


,script_id,imdb_id,title,year,gross_infl_adj_musd,total_words,f_words,n_characters,n_female,pct_female_dialogue,pct_female_characters,female_lead
0,1534,tt1022603,(500) Days of Summer,2009,37.0,18500,5738,13,5,0.310162,0.384615,False
1,1512,tt0147800,10 Things I Hate About You,1999,65.0,19680,8992,12,4,0.456911,0.333333,True
2,1514,tt0417385,12 and Holding,2005,NaN,15968,5324,15,5,0.333417,0.333333,False


## Phase 4a — normalize the Bechdel key

Mirror `imdb_id` arrives with the `tt` prefix and leading zeros stripped
(e.g. `3` for tt0000003). Pudding's is already `tt` + 7 digits. Normalize the
Bechdel side: zero-pad to 7, prepend `tt`.

In [6]:
bechdel = bechdel.dropna(subset=["imdb_id"]).copy()
bechdel["imdb_id_norm"] = "tt" + bechdel["imdb_id"].str.zfill(7)

# sanity: normalized ids must look like tt + >=7 digits, and be unique
bad = ~bechdel["imdb_id_norm"].str.fullmatch(r"tt\d{7,8}")
print(f"malformed after normalization: {bad.sum()}")
if bad.sum():
    display(bechdel[bad].head())
    bechdel = bechdel[~bad]

print("normalized id unique:", bechdel["imdb_id_norm"].is_unique)
dupe_b = bechdel[bechdel["imdb_id_norm"].duplicated(keep=False)]
if len(dupe_b):
    display(dupe_b.sort_values("imdb_id_norm").head(10))
    bechdel = bechdel.drop_duplicates("imdb_id_norm", keep="first")
print(f"bechdel ready: {len(bechdel):,} films")

malformed after normalization: 2


,year,id,imdb_id,title,rating,imdb_id_norm
7229,2014,8702,002180411,Into the Woods,3,tt002180411
7492,2015,8703,001226766,The Great Gilly Hopkins,3,tt001226766


normalized id unique: False


,year,id,imdb_id,title,rating,imdb_id_norm
435,1942,5939,0035279,Saboteur,1,tt0035279
742,1956,5938,0035279,Saboteur,1,tt0035279
840,1959,9209,0053285,Sleeping Beauty,3,tt0053285
848,1959,474,53285,Sleeping Beauty,3,tt0053285
1782,1983,4449,0086425,Terms of Endearment,3,tt0086425
1792,1983,4448,0086425,Terms of Endearment,1,tt0086425
2914,1997,4380,0117056,Ayneh,3,tt0117056
2998,1997,4381,0117056,Ayneh,3,tt0117056
6010,2011,4889,2043900,Last Call at the Oasis,3,tt2043900
6092,2011,4907,2043900,Last Call at the Oasis,3,tt2043900


bechdel ready: 8,827 films


## Phase 4b — the clean join + window filter

In [7]:
spine = pudding_film.merge(
    bechdel[["imdb_id_norm", "rating", "year"]].rename(
        columns={"rating": "bechdel_rating", "year": "bechdel_year"}),
    left_on="imdb_id", right_on="imdb_id_norm", how="inner",
).drop(columns="imdb_id_norm")

spine["bechdel_pass"] = spine["bechdel_rating"].eq(3)

in_window = spine["year"].between(1990, 2015)
print(f"pudding_film: {len(pudding_film):,}")
print(f"matched to Bechdel: {len(spine):,} ({len(spine)/len(pudding_film):.1%})")
print(f"in 1990-2015 window: {in_window.sum():,}")

spine = spine[in_window].reset_index(drop=True)

pudding_film: 2,000
matched to Bechdel: 1,476 (73.8%)
in 1990-2015 window: 1,189


## Phase 4c — validate the join

1. **Year agreement** — matched pairs should have (nearly) the same year on
   both sides; big disagreements would mean the id itself is wrong.
2. **Who didn't match** — do unmatched Pudding films differ systematically
   (year, gross) from matched ones? That's the selection check.
3. **Rating distribution** — sane proportions vs. the full Bechdel DB.

In [8]:
yd = (spine["year"] - spine["bechdel_year"]).abs()
print("year agreement: ", (yd <= 1).mean().round(4), "within +/-1")
print("disagreements > 1yr:", (yd > 1).sum())
if (yd > 1).any():
    display(spine.loc[yd > 1, ["imdb_id", "title", "year", "bechdel_year", "bechdel_rating"]].head(10))

year agreement:  0.9992 within +/-1
disagreements > 1yr: 1


,imdb_id,title,year,bechdel_year,bechdel_rating
941,tt2403021,The Green Inferno,2013,2015,3


In [9]:
pud_w = pudding_film[pudding_film["year"].between(1990, 2015)]
unmatched = pud_w[~pud_w["imdb_id"].isin(spine["imdb_id"])]
print(f"window Pudding films: {len(pud_w):,} | matched: {len(spine):,} "
      f"({len(spine)/len(pud_w):.1%}) | unmatched: {len(unmatched):,}")
print("\nmatched vs unmatched (selection check):")
cmp = pd.DataFrame({
    "matched": spine[["year", "gross_infl_adj_musd", "pct_female_dialogue"]].mean(),
    "unmatched": unmatched[["year", "gross_infl_adj_musd", "pct_female_dialogue"]].mean(),
}).round(3)
print(cmp)
print("\nunmatched sample (eyeball — obscure? foreign? recent?):")
unmatched[["imdb_id", "title", "year", "gross_infl_adj_musd"]].sample(min(10, len(unmatched)), random_state=530)

window Pudding films: 1,587 | matched: 1,189 (74.9%) | unmatched: 398

matched vs unmatched (selection check):
                      matched  unmatched
year                 2004.805   2001.701
gross_infl_adj_musd   105.874     41.863
pct_female_dialogue     0.300      0.258

unmatched sample (eyeball — obscure? foreign? recent?):


,imdb_id,title,year,gross_infl_adj_musd
931,tt0109836,Mary Shelley's Frankenstein,1994,45.0
1100,tt3733778,Pay the Ghost,2015,NaN
576,tt0118900,Gang Related,1997,10.0
1424,tt0365737,Syriana,2005,68.0
1545,tt0101761,The Doors,1991,72.0
1166,tt0102753,Rambling Rose,1991,12.0
1456,tt2216212,The Amityville Asylum,2013,NaN
1961,tt2231253,Wild Card,2015,NaN
1512,tt0318374,The Cooler,2003,11.0
791,tt0251160,John Q,2002,106.0


In [10]:
print("bechdel_rating distribution in spine vs full Bechdel DB (1990-2015):")
b_w = bechdel[bechdel["year"].between(1990, 2015)]
dist = pd.DataFrame({
    "spine": spine["bechdel_rating"].value_counts(normalize=True).sort_index(),
    "full_db_window": b_w["rating"].value_counts(normalize=True).sort_index(),
}).round(3)
print(dist)
print(f"\nspine pass rate (rating==3): {spine['bechdel_pass'].mean():.1%}")

bechdel_rating distribution in spine vs full Bechdel DB (1990-2015):
   spine  full_db_window
0  0.063           0.079
1  0.304           0.225
2  0.116           0.091
3  0.517           0.604

spine pass rate (rating==3): 51.7%


In [11]:
out_path = INTERIM / "spine.csv"
spine.to_csv(out_path, index=False)
print(f"wrote {len(spine):,} rows x {spine.shape[1]} cols -> {out_path}")
spine.head(3)

wrote 1,189 rows x 15 cols -> /Users/trevorjmckee/Documents/Work/School/03_Summer_2026/DSC530/final_project/data/interim/spine.csv


,script_id,imdb_id,title,year,gross_infl_adj_musd,total_words,f_words,n_characters,n_female,pct_female_dialogue,pct_female_characters,female_lead,bechdel_rating,bechdel_year,bechdel_pass
0,1534,tt1022603,(500) Days of Summer,2009,37.0,18500,5738,13,5,0.310162,0.384615,False,1,2009,False
1,1512,tt0147800,10 Things I Hate About You,1999,65.0,19680,8992,12,4,0.456911,0.333333,True,3,1999,True
2,1514,tt0417385,12 and Holding,2005,NaN,15968,5324,15,5,0.333417,0.333333,False,3,2005,True


## Findings log

- **Match rate:** 1,189 of 1,587 window Pudding films matched Bechdel (**74.9%**) —
  NOT the near-lossless join the plan predicted. The 398 unmatched films are ones
  bechdeltest.com volunteers never rated (as of the Mar-2021 mirror).
- **Selection check — this matters:** unmatched films are much smaller
  (mean infl-adj gross **$42M vs $106M** matched) and have somewhat lower female
  dialogue share (**25.8% vs 30.0%**). The Bechdel join therefore tightens the
  sample toward bigger films — a second layer of range restriction on top of
  Pudding's own pre-filter. First-class limitation, and a reason to re-try the
  live API before final submission (post-2021 ratings would recover some of the 398).
- **Year agreement:** 99.9% within ±1 (single >1yr disagreement, displayed above) —
  the imdb_id key itself is sound.
- **Rating distribution:** spine pass rate 51.7% vs 60.4% in the full Bechdel DB
  window — the spine skews slightly toward rating-1 films relative to the DB.
- **Exclusions applied:** 5 gender-'?' characters; 0 zero-word characters;
  2 malformed + 12 duplicate Bechdel ids dropped. meta imdb_id was already unique.
- Spine uses the 2021 Bechdel MIRROR (no `dubious` flag; post-2021 ratings absent).